# Olfactory Intelligence Benchmark csv file

In [1]:
import uuid
from pathlib import Path
from typing import List, Tuple
import pandas as pd


ROOT = Path(".").resolve()              
OUT_CSV = Path("OP_Benchmark.csv")  
if OUT_CSV.exists():
    OUT_CSV.unlink()
DEDUPE = False                         

REQUIRED_COLS = [
    "question_ID",
    "compound.name_1",
    "SMILES_1",
    "compound.name_2",
    "SMILES_2",
    "OPTIONS",
    "question_category",
    "prompt.1",
    "prompt.2",
    "answer",
    "other_info",
]


def find_csvs_one_level(root: Path) -> List[Path]:
    """
    Return CSV files located in immediate subfolders.
    Does not include CSV files in `root` itself.
    """
    results: List[Path] = []
    for entry in root.iterdir():
        if entry.is_dir():
            results.extend(entry.glob("*.csv"))
    return results

def validate_columns(df: pd.DataFrame) -> Tuple[bool, List[str]]:
    cols = list(df.columns.astype(str))
    missing = [c for c in REQUIRED_COLS if c not in cols]
    return (len(missing) == 0), missing

candidates = find_csvs_one_level(ROOT)

good_frames: List[pd.DataFrame] = []
included_paths: List[Path] = []
excluded_missing: List[Tuple[Path, List[str]]] = []
excluded_errors: List[Tuple[Path, str]] = []

for csv_path in candidates:
    try:
        df = pd.read_csv(csv_path, dtype=str, keep_default_na=False, na_values=[])
    except Exception as e:
        excluded_errors.append((csv_path, f"read error: {e}"))
        print(f"{csv_path} | read error: {e}")
        continue

    ok, missing = validate_columns(df)
    if not ok:
        excluded_missing.append((csv_path, missing))
        print(f"{csv_path} | missing columns: {', '.join(missing)}")
        continue

    # keep only required columns, in exact order
    df = df[REQUIRED_COLS].copy()
    good_frames.append(df)
    included_paths.append(csv_path)
    

if not good_frames:
    print("\nNo valid CSVs matched the required headers. Nothing to combine.")
else:
    combined = pd.concat(good_frames, axis=0, ignore_index=True)

    if DEDUPE:
        before = len(combined)
        combined = combined.drop_duplicates()
        print(f"\nDe-duplicated rows: {before} → {len(combined)}")

    # ensure exact column order
    combined = combined[REQUIRED_COLS]

    # Replace question_ID values with fresh UUID4 strings (universal identifiers)
    combined["question_ID"] = [str(uuid.uuid4()) for _ in range(len(combined))]

    # write output
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    combined.to_csv(OUT_CSV, index=False)

    # summary
    print("\nSummary")
    print("-------")
    print(f"Included files: {len(included_paths)}")
    print(f"Total combined rows: {len(combined)}")
    print(f"Done")

    if excluded_missing:
        print("\nFiles excluded due to missing columns:")
        for p, missing in excluded_missing:
            print(f"  - {p}  (missing: {', '.join(missing)})")

    if excluded_errors:
        print("\nFiles excluded due to read errors:")
        for p, msg in excluded_errors:
            print(f"  - {p}  ({msg})")



Summary
-------
Included files: 8
Total combined rows: 1010
Done
